# Dual-Stream LLM Generation Example


In [2]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoConfig
from datasets import load_dataset
from IPython.display import display, Markdown
import random
from tqdm import tqdm
from model import DualStreamLLM

# --- Configuration ---
run_dir = 'checkpoints/run_20251228_230459'
checkpoint_path = f'{run_dir}/checkpoint-120000/model.pt'
config_path = f'{run_dir}/training_config.json'
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
num_examples = 10
max_new_tokens = 50
temperature = 0.1  # Low temperature for more coherent generation

print(f"Using device: {device}")
print(f"Loading checkpoint: {checkpoint_path}")


Using device: cuda:0
Loading checkpoint: checkpoints/run_20251228_230459/checkpoint-120000/model.pt


In [3]:
device = 'cuda:1' if torch.cuda.is_available() else 'cpu'
device

'cuda:1'

In [4]:
# Load training configuration
with open(config_path, 'r') as f:
    config = json.load(f)



# Get model parameters from config
base_model_config = AutoConfig.from_pretrained(config['model_name'])
vocab_size = base_model_config.vocab_size
hidden_dim = base_model_config.hidden_size

# Instantiate the model
model = DualStreamLLM(
    model_name=config['model_name'],
    freeze_backbone=config['freeze_backbone'],
    embedding_init_strategy=config['embedding_init_strategy'],
    unembedding_init_strategy=config.get('unembedding_init_strategy'),
    embedding_init_noise_std=config['embedding_init_noise_std'],
    objective=config['objective'],
    shared_embeddings=config['shared_embeddings']
)

print("Model instantiated successfully.")
print(f"  - Objective: {config['objective']}")
print(f"  - Shared Embeddings: {config['shared_embeddings']}")



Model instantiated successfully.
  - Objective: separate_distillation
  - Shared Embeddings: True


In [5]:
# Load the checkpoint
checkpoint = torch.load(checkpoint_path, map_location='cpu')

# The model was trained with FSDP, so we need to clean the state dict keys
# by removing the '_fsdp_wrapped_module.' prefix.
model_state_dict = checkpoint['model_state_dict']
cleaned_state_dict = {key.replace('_fsdp_wrapped_module.', ''): value for key, value in model_state_dict.items()}

# Load the cleaned state dict
model.load_state_dict(cleaned_state_dict)

# Set model to evaluation mode and move to device
model.to(device)
model.eval()

print("Model weights loaded successfully.")


/tmp/ipykernel_713860/447543396.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu')


OutOfMemoryError: CUDA out of memory. Tried to allocate 246.00 MiB. GPU 1 has a total capacity of 79.26 GiB of which 73.19 MiB is free. Process 752106 has 78.77 GiB memory in use. Including non-PyTorch memory, this process has 414.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Load dataset (fineweb doesn't have validation split, use train with skip)
dataset = load_dataset(config['dataset_name'], split='train', streaming=True)
dataset = dataset.shuffle(seed=42, buffer_size=10000)

# Skip training samples (120000 steps * batch_size=8 * 2 for pairs)
num_skip = 120000 * 8 * 2
print(f"Skipping {num_skip:,} training samples...")
dataset_iter = iter(dataset)
for _ in tqdm(range(num_skip), desc="Skipping train samples"):
    next(dataset_iter)

# Get samples for generation (2 per example for A and B prompts)
samples = [next(dataset_iter) for _ in range(num_examples * 2)]
print(f"Loaded {len(samples)} samples for generation.")


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Skipping 1,920,000 training samples...


Skipping train samples: 100%|██████████| 1920000/1920000 [07:35<00:00, 4217.70it/s]

Loaded 20 samples for generation.


In [ ]:
def generate(input_ids_A, input_ids_B):
    """
    Generates two sequences of text from two token sequences using the dual-stream model.
    Both inputs must have the same length (no padding).
    """
    assert input_ids_A.shape == input_ids_B.shape, f"Input shapes must match: {input_ids_A.shape} vs {input_ids_B.shape}"
    
    with torch.no_grad():
        input_ids_A = input_ids_A.to(device)
        input_ids_B = input_ids_B.to(device)
        attention_mask_A = torch.ones_like(input_ids_A)
        attention_mask_B = torch.ones_like(input_ids_B)
        
        # Generation loop
        for _ in range(max_new_tokens):
            logits_A, logits_B = model(input_ids_A, attention_mask_A, input_ids_B, attention_mask_B)

            # Get logits for the last token and apply temperature
            logits_A_scaled = logits_A[:, -1, :] / temperature
            logits_B_scaled = logits_B[:, -1, :] / temperature
            
            # Sample next token
            probs_A = F.softmax(logits_A_scaled, dim=-1)
            probs_B = F.softmax(logits_B_scaled, dim=-1)
            next_token_A = torch.multinomial(probs_A, num_samples=1)
            next_token_B = torch.multinomial(probs_B, num_samples=1)
            
            # Append next token to the input
            input_ids_A = torch.cat([input_ids_A, next_token_A], dim=1)
            input_ids_B = torch.cat([input_ids_B, next_token_B], dim=1)
            attention_mask_A = torch.cat([attention_mask_A, torch.ones((1, 1), device=device, dtype=torch.long)], dim=1)
            attention_mask_B = torch.cat([attention_mask_B, torch.ones((1, 1), device=device, dtype=torch.long)], dim=1)

            # Stop if both streams generate EOS token
            if next_token_A.item() == tokenizer.eos_token_id and next_token_B.item() == tokenizer.eos_token_id:
                break
                
        generated_A = tokenizer.decode(input_ids_A[0], skip_special_tokens=True)
        generated_B = tokenizer.decode(input_ids_B[0], skip_special_tokens=True)
        
        return generated_A, generated_B


In [ ]:

def generate_from_texts(text_A, text_B, prompt_len_tokens=40, error_on_min_length=True):

    tokens_A = tokenizer.encode(text_A, return_tensors='pt')
    tokens_B = tokenizer.encode(text_B, return_tensors='pt')
    
    
    if error_on_min_length and (tokens_A.shape[1] < prompt_len_tokens or tokens_B.shape[1] < prompt_len_tokens):
        raise ValueError(f"Text is too short: A={tokens_A.shape[1]} tokens, B={tokens_B.shape[1]} tokens (minimum {prompt_len_tokens})")
    
    input_ids_A = tokens_A[:, :prompt_len_tokens]
    input_ids_B = tokens_B[:, :prompt_len_tokens]
    
    prompt_A = tokenizer.decode(input_ids_A[0], skip_special_tokens=True)
    prompt_B = tokenizer.decode(input_ids_B[0], skip_special_tokens=True)
    
    generated_A, generated_B = generate(input_ids_A, input_ids_B)
    
    newly_generated_A = generated_A[len(prompt_A):].strip() if generated_A.startswith(prompt_A) else generated_A
    newly_generated_B = generated_B[len(prompt_B):].strip() if generated_B.startswith(prompt_B) else generated_B
    
    return prompt_A, newly_generated_A, prompt_B, newly_generated_B

In [ ]:
PROMPT_LEN_TOKENS = 40

# Generate and print examples
for i in range(num_examples):
    text_A = samples[i * 2]['text']
    text_B = samples[i * 2 + 1]['text']
    
    try:
        prompt_A, generated_A, prompt_B, generated_B = generate_from_texts(text_A, text_B, PROMPT_LEN_TOKENS)
        
        print(f"### Example {i+1}/{num_examples}")
        print(f"**Stream A:**")
        print(f"[PROMPT] {prompt_A}")
        print(f"[GENERATED] {generated_A}")
        print()
        print(f"**Stream B:**")
        print(f"[PROMPT] {prompt_B}")
        print(f"[GENERATED] {generated_B}")
        print(f"---\n")
    except ValueError as e:
        print(f"Skipping example {i+1}: {e}")

### Example 1/10
**Stream A:**
[PROMPT] Potential pattern found in medical malpractice payouts
A recently released study provides new insights into the world of medical errors and even indicates which types of providers are most often involved in claims.
New York
[GENERATED] University's Department of Medicine and the University of Pennsylvania's School of Medicine have published a study that shows that the number of medical malpractice claims filed by physicians and other medical professionals has increased by more than 50 percent in the past decade.

**Stream B:**
[PROMPT] Passionate about fine quality underwear, Jockey Underwear is the brand every man turns to for those much-needed underwear drawer upgrades. Crafted from the best quality fabrics with incredible attention to detail,
[GENERATED] and the most popular quality of the world.
---

### Example 2/10
**Stream A:**
[PROMPT] Team Ninja Warrior, a spin-off of the popular show American Ninja Warrior, is in the works for the Esqui

In [ ]:
prefix = "Possible Categories: Art, Business, Cooking, History, Music, Politics, Science, Sports, Technology, Travel.\n\n"
body_a = "Text: The tropical beaches of Bali offer stunning sunsets and a beatiful crystal clear water."
body_b = "Text: The senator proposed a new bill to reform healthcare policy."
postfix = '\nThe Category of the above text is'


max_prefix_body_tokens = 41
tokens_a = tokenizer.encode(prefix + body_a, return_tensors='pt')
tokens_b = tokenizer.encode(prefix + body_b, return_tensors='pt')
text_a_trunc = tokenizer.decode(tokens_a[0, :max_prefix_body_tokens], skip_special_tokens=True) + postfix
text_b_trunc = tokenizer.decode(tokens_b[0, :max_prefix_body_tokens], skip_special_tokens=True) + postfix

# print(text_a_trunc)
# print('---')
# print(text_b_trunc)
generate_from_texts(
    text_a_trunc,
    text_b_trunc,
    prompt_len_tokens=100,
    error_on_min_length=False,
 )

NameError: name 'tokenizer' is not defined